In [8]:
import pandas as pd
import json
import ast
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "ml").is_dir() and (p / "requirements.txt").exists():
            return p
    return start

REPO_ROOT = _find_repo_root(Path.cwd())
ML_DIR = REPO_ROOT / "ml"
DATA_DIR = ML_DIR / "data"

if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))


In [9]:
df = pd.read_csv(DATA_DIR / "kaggle_prepared.csv")

In [10]:
number_of_users = df['user_id'].nunique()
print(f"Liczba unikalnych użytkowników: {number_of_users}")

Liczba unikalnych użytkowników: 206209


In [11]:
min_orders = 9
valid_users = [uid for uid, grp in df.groupby("user_id") if len(grp) >= min_orders]
n_samples = 5
step = max(1, len(valid_users) // n_samples)
chosen = valid_users[::step][:n_samples]

In [12]:
def _parse_categories(raw) -> list[str]:
    # Zachowujemy oryginalną pisownię kategorii (bez .lower()) – kategorie są kanoniczne.
    if isinstance(raw, list):
        return [str(c).strip() for c in raw if c]
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(c).strip() for c in parsed if c]
    except Exception:
        pass
    return [raw.strip()]

def _parse_product_ids(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(x) for x in raw if pd.notna(x)]
    if pd.isna(raw):
        return []
    if isinstance(raw, str):
        text = raw.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if pd.notna(x)]
        except Exception:
            pass
        return [text]
    return [str(raw)]

# Wczytanie mapowania ID -> nazwa produktu + kategorie (OpenFoodData)
df2 = pd.read_csv(DATA_DIR / "OpenFoodData.csv", usecols=["Id", "Name", "Categories"])
df2["Id"] = df2["Id"].astype(str)
df2 = df2.dropna(subset=["Name"])
df2["Name"] = df2["Name"].astype(str).str.strip()
df2["Categories_parsed"] = df2["Categories"].apply(_parse_categories)
id2name: dict[str, str] = dict(zip(df2["Id"], df2["Name"]))
id2categories: dict[str, list[str]] = dict(zip(df2["Id"], df2["Categories_parsed"]))

# Ta komórka ma TYLKO zbudować i wypisać JSON wejściowy (bez RAG/testów).
for i, uid in enumerate(chosen, 1):
    print(f"\n{'─'*55}")
    print(f"[JSON {i}/{len(chosen)}] user_id={uid}")
    user_df = df[df["user_id"] == uid].sort_values("order_id").reset_index(drop=True)
    history_rows = user_df.iloc[:-1].reset_index(drop=True)

    # "date" ma oznaczać liczbę dni od ostatnich zakupów (ostatniego koszyka w historii).
    gaps = history_rows["days_since_prior_order"].fillna(0).astype(int).tolist()
    recency_days = []
    suffix = 0
    for gap in reversed(gaps):
        recency_days.append(int(suffix))
        suffix += int(gap)
    recency_days = list(reversed(recency_days))

    history_baskets = {"history_baskets": []}
    for idx, row in history_rows.iterrows():
        products = []
        row_product_ids = _parse_product_ids(row["off_product_id"])
        for pid in row_product_ids:
            pid = str(pid)
            products.append({
                "name": id2name.get(pid, pid),
                "id": pid,
                "categories": id2categories.get(pid, []),
            })
        days = recency_days[idx] if idx < len(recency_days) else 0
        history_baskets["history_baskets"].append({"date": days, "products": products})

    history_baskets_str = json.dumps(history_baskets, ensure_ascii=False)
    print(history_baskets_str)


───────────────────────────────────────────────────────
[JSON 1/5] user_id=1
{"history_baskets": [{"date": 143, "products": [{"name": "Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g", "id": "5902802802576", "categories": ["Dodatki do żywności", "Środki przeciwzbrylające", "Wodorowęglany sodu"]}, {"name": "Beef Jerky – Tarczyński", "id": "5908230536007", "categories": ["Suszona wołowina"]}, {"name": "Pistachio Delight – Carte d’Or", "id": "8711327672499", "categories": ["Desery", "Mrożonki", "Desery mrożone", "Lody i sorbety", "Lody", "Lody w pudełku"]}, {"name": "Ser włoszczowski – Auchan", "id": "2228326001442", "categories": ["Produkty mleczne", "Produkty fermentowane", "Produkty mleczne fermentowane", "Sery"]}, {"name": "Jabłka – La-Sad – 1,5 kg", "id": "20355944", "categories": ["Produkty roślinne", "Produkty na bazie owoców i warzyw", "Produkty na bazie owoców", "Owoce", "Jabłka"]}, {"name": "Suszone jabłko o smaku ananasa – Crispy natural – 18g", "id": "5900238965421", "ca

In [13]:
def _parse_categories(raw) -> list[str]:
    # Zachowujemy oryginalną pisownię kategorii (bez .lower()) – kategorie są kanoniczne.
    if isinstance(raw, list):
        return [str(c).strip() for c in raw if c]
    if not isinstance(raw, str) or not raw.strip():
        return []
    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            return [str(c).strip() for c in parsed if c]
    except Exception:
        pass
    return [raw.strip()]

def _parse_product_ids(raw) -> list[str]:
    if isinstance(raw, list):
        return [str(x) for x in raw if pd.notna(x)]
    if pd.isna(raw):
        return []
    if isinstance(raw, str):
        text = raw.strip()
        if not text:
            return []
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return [str(x) for x in parsed if pd.notna(x)]
        except Exception:
            pass
        return [text]
    return [str(raw)]

df2 = pd.read_csv(DATA_DIR / "OpenFoodData.csv", usecols=["Id", "Name", "Categories"])
df2["Id"] = df2["Id"].astype(str)
df2 = df2.dropna(subset=["Name"])
_product_cache = {}
for _, row in df2.iterrows():
    pid = row["Id"]
    _product_cache[pid] = {
        "name": str(row["Name"]).strip(),
        "categories": _parse_categories(row.get("Categories", "")),
    }

recalls = []
for i, uid in enumerate(chosen, 1):
        print(f"\n{'─'*55}")
        print(f"[EVAL {i}/{len(chosen)}] user_id={uid}")
        id2name: dict[str, str] = {pid: info["name"] for pid, info in _product_cache.items()}
        user_df = df[df["user_id"] == uid].sort_values("order_id").reset_index(drop=True)
        history_rows = user_df.iloc[:-1].reset_index(drop=True)
        target_rows = user_df.iloc[-1]
        target = []
        target_ids = _parse_product_ids(target_rows["off_product_id"])
        for x in target_ids:
            if x in id2name:
                target.append(id2name[x])

        # "date" ma oznaczać liczbę dni od ostatnich zakupów (ostatniego koszyka w historii).
        gaps = history_rows["days_since_prior_order"].fillna(0).astype(int).tolist()
        recency_days = []
        suffix = 0
        for gap in reversed(gaps):
            recency_days.append(int(suffix))
            suffix += int(gap)
        recency_days = list(reversed(recency_days))

        history_baskets = {"history_baskets": []}
        for idx, row in history_rows.iterrows():
            products = []
            row_product_ids = _parse_product_ids(row["off_product_id"])
            for x in row_product_ids:
                pid = str(x)
                info = _product_cache.get(pid)
                name = info["name"] if info else pid
                categories = info["categories"] if info else []
                products.append({"name": name, "id": pid, "categories": categories})
            days = recency_days[idx] if idx < len(recency_days) else 0
            history_baskets["history_baskets"].append({"date": days, "products": products})

        history_baskets_str = json.dumps(history_baskets, ensure_ascii=False)
        print(history_baskets_str)
        import rag_pipeline
        import importlib
        importlib.reload(rag_pipeline)
        from rag_pipeline import stage1, stage2, stage3
        res_stage1 = stage1(history_baskets_str)
        print(res_stage1)
        res_stage2 = stage2(res_stage1)
        print(res_stage2)
        res_stage3 = stage3(res_stage2, history_baskets_str, final_k=10)
        print(res_stage3)
        recall = 0
        for r in res_stage3['recommended_products']:
            if r in target:
                recall += 1
        recalls.append(recall / len(target) if target else 0.0)
        print(recalls[-1])
res = sum(recalls) / len(recalls) if recalls else 0.0
print(f"Recall@10: {res}")


───────────────────────────────────────────────────────
[EVAL 1/5] user_id=1
{"history_baskets": [{"date": 143, "products": [{"name": "Soda oczyszczona spożywcza bez antyzbrylaczy – 1000g", "id": "5902802802576", "categories": ["Dodatki do żywności", "Środki przeciwzbrylające", "Wodorowęglany sodu"]}, {"name": "Beef Jerky – Tarczyński", "id": "5908230536007", "categories": ["Suszona wołowina"]}, {"name": "Pistachio Delight – Carte d’Or", "id": "8711327672499", "categories": ["Desery", "Mrożonki", "Desery mrożone", "Lody i sorbety", "Lody", "Lody w pudełku"]}, {"name": "Ser włoszczowski – Auchan", "id": "2228326001442", "categories": ["Produkty mleczne", "Produkty fermentowane", "Produkty mleczne fermentowane", "Sery"]}, {"name": "Jabłka – La-Sad – 1,5 kg", "id": "20355944", "categories": ["Produkty roślinne", "Produkty na bazie owoców i warzyw", "Produkty na bazie owoców", "Owoce", "Jabłka"]}, {"name": "Suszone jabłko o smaku ananasa – Crispy natural – 18g", "id": "5900238965421", "ca